# Structured & SQL RAG — A Practical Guide for Beginners

**Patria & Co. 2026** · [www.patriaco.co.uk](https://www.patriaco.co.uk)

*A huge amount of company knowledge lives inside a **database**, not inside documents. This notebook shows how to answer business questions with a different pipeline: **fetch the schema → write SQL → run it → answer**.*

## What you'll build

Most RAG (Retrieval-Augmented Generation) systems treat knowledge as **text**: documents are split into paragraphs, turned into embeddings, matched against a question by similarity, and read by a model. That works great when the answer is **already written down** somewhere.

But try asking: *"What was our total revenue from shipped orders?"*
No paragraph anywhere contains that number. It only **exists** once we:

1. JOIN the `orders` table with the `products` table,
2. filter to rows where the status is `shipped`,
3. multiply `price × quantity` for each row,
4. add it all up.

Similarity search over text (dense retrieval) is great at finding **text with similar meaning**, but it **cannot do arithmetic**. So for databases we swap the pipeline:

| Pipeline for documents | Pipeline for databases (SQL RAG) |
|---|---|
| embed → retrieve → ground → generate | **retrieve schema → generate SQL → execute → answer** |

In this notebook you will build, step by step:

- A tiny **SQLite** database with three tables (`customers`, `products`, `orders`) — similar to what might sit behind an online store's chatbot.
- **Schema cards** and a **domain glossary** — short summaries of each table, plus a dictionary mapping business terms to database columns.
- A `retrieve_schema()` function — picks the relevant tables (**schema linking**).
- A `generate_sql()` function — produces SQL that can **actually be executed**.
- A `sql_rag()` function — the full pipeline: run the SQL and return an answer.
- A `route()` function — decides whether a question needs SQL (numbers) or documents (descriptions).

> **Runs entirely offline.** No internet connection and no API key needed. What gets **really executed** is SQL against a genuine SQLite database. Two parts are intentionally simplified for teaching purposes: (1) table selection uses a simple word-overlap score, and (2) SQL is produced with hand-written rules instead of an LLM. The overall control flow, however, is identical to a production system.


## Setup

We only need two modules from Python's *standard library*:

- `re` — to split text into words (tokenizing).
- `sqlite3` — a real database engine that runs in memory, no installation required.


In [1]:
import re
import sqlite3

print(f"SQLite version {sqlite3.sqlite_version} is ready.")
print("No internet connection or API key needed.")


SQLite version 3.45.1 is ready.
No internet connection or API key needed.


## Step 1 — Create a small database

This is the kind of knowledge that lives in a **database**, not a document: it gets *queried* and aggregated, not read. We'll build three tables that might sit behind an online store's support bot:

- **`customers`** — one row per customer, with a `tier` column: `free`, `pro`, or `enterprise`.
- **`products`** — one row per product, with a `price` (unit price).
- **`orders`** — one row per order, linking a customer to a product, plus `quantity` and `status` (`shipped` / `refunded` / `pending`).

> **Notice:** there is **no `revenue` column anywhere!** Revenue has to be **computed** from `price × quantity`. This is exactly why we need SQL RAG in the first place.


In [2]:
def build_db():
    """Create an in-memory SQLite database and fill it with sample data."""
    db = sqlite3.connect(":memory:")

    # 1. Schema definition (CREATE TABLE)
    db.executescript("""
        CREATE TABLE customers (
            id      INTEGER PRIMARY KEY,
            name    TEXT,
            country TEXT,
            tier    TEXT      -- 'free' | 'pro' | 'enterprise'
        );
        CREATE TABLE products (
            id       INTEGER PRIMARY KEY,
            name     TEXT,
            category TEXT,
            price    REAL
        );
        CREATE TABLE orders (
            id          INTEGER PRIMARY KEY,
            customer_id INTEGER,
            product_id  INTEGER,
            quantity    INTEGER,
            status      TEXT,     -- 'shipped' | 'refunded' | 'pending'
            order_date  TEXT      -- ISO format: YYYY-MM-DD
        );
    """)

    # 2. Fill the customers table
    db.executemany("INSERT INTO customers VALUES (?,?,?,?)", [
        (1, "Ada",    "DE", "pro"),
        (2, "Bashir", "TR", "enterprise"),
        (3, "Chen",   "SG", "free"),
        (4, "Dilan",  "TR", "pro"),
        (5, "Eka",    "ID", "enterprise"),
        (6, "Farid",  "ID", "free"),
    ])

    # 3. Fill the products table
    db.executemany("INSERT INTO products VALUES (?,?,?,?)", [
        (10, "Aurora Lamp",    "lighting", 49.0),
        (11, "Nimbus Speaker", "audio",    129.0),
        (12, "Coil Cable",     "audio",    9.0),
        (13, "Solaris Bulb",   "lighting", 19.0),
    ])

    # 4. Fill the orders table
    db.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [
        (100, 1, 11, 2, "shipped",  "2026-03-04"),
        (101, 2, 10, 1, "shipped",  "2026-03-09"),
        (102, 2, 12, 5, "refunded", "2026-03-12"),
        (103, 4, 11, 1, "shipped",  "2026-03-20"),
        (104, 3, 10, 3, "pending",  "2026-03-28"),
        (105, 5, 13, 4, "shipped",  "2026-04-02"),
        (106, 5, 11, 1, "shipped",  "2026-04-10"),
        (107, 6, 12, 2, "pending",  "2026-04-15"),
    ])

    db.commit()
    return db


# Build the database and check row counts
db = build_db()
print("Row count per table:")
print(f"  customers = {db.execute('SELECT COUNT(*) FROM customers').fetchone()[0]}")
print(f"  products  = {db.execute('SELECT COUNT(*) FROM products').fetchone()[0]}")
print(f"  orders    = {db.execute('SELECT COUNT(*) FROM orders').fetchone()[0]}")
print()
print("Note: there is no 'revenue' column — it must be computed from price * quantity.")


Row count per table:
  customers = 6
  products  = 4
  orders    = 8

Note: there is no 'revenue' column — it must be computed from price * quantity.


## Step 2 — Schema cards and Domain Glossary

### Schema card
A **short description of each table**: its name, its columns, and one or two sentences explaining what a single row means. This is what we'll **retrieve** — not a document, not a paragraph. In a production system, these cards are embedded with a language model.

### Domain glossary
A **dictionary that maps business terms to database columns**. Users often ask about "revenue", "active users", "churn" — business words that are **not** actual column names. Without a glossary, a model might invent a column that doesn't exist. With a glossary, it knows that "revenue" really means `SUM(price × quantity)`.


In [3]:
# --- Schema cards: one card per table ---
SCHEMA_CARDS = {
    "customers": {
        "columns": ["id", "name", "country", "tier"],
        "doc": "One row per customer: name, country code, and subscription tier "
               "(free, pro, enterprise). Use this for questions about customer counts.",
    },
    "products": {
        "columns": ["id", "name", "category", "price"],
        "doc": "One row per product: name, category, and unit price. "
               "Revenue is computed from price here times quantity in orders.",
    },
    "orders": {
        "columns": ["id", "customer_id", "product_id", "quantity", "status", "order_date"],
        "doc": "One row per order: which customer bought which product, how many, "
               "the status (shipped, refunded, pending), and the date. Revenue and "
               "refund counts both come from this table.",
    },
}

# --- Domain glossary: business term -> database column/value ---
GLOSSARY = {
    "revenue":              "SUM(products.price * orders.quantity), there is no 'revenue' column",
    "enterprise customer":  "customers.tier = 'enterprise'",
    "refund / refunded":    "orders.status = 'refunded'",
    "shipped":              "orders.status = 'shipped'",
    "pending":              "orders.status = 'pending'",
}

print("=== SCHEMA CARDS ===")
for table, card in SCHEMA_CARDS.items():
    print(f"\n[{table}] columns: {card['columns']}")
    print(f"  -> {card['doc']}")

print("\n=== DOMAIN GLOSSARY ===")
for term, meaning in GLOSSARY.items():
    print(f"  '{term}' = {meaning}")


=== SCHEMA CARDS ===

[customers] columns: ['id', 'name', 'country', 'tier']
  -> One row per customer: name, country code, and subscription tier (free, pro, enterprise). Use this for questions about customer counts.

[products] columns: ['id', 'name', 'category', 'price']
  -> One row per product: name, category, and unit price. Revenue is computed from price here times quantity in orders.

[orders] columns: ['id', 'customer_id', 'product_id', 'quantity', 'status', 'order_date']
  -> One row per order: which customer bought which product, how many, the status (shipped, refunded, pending), and the date. Revenue and refund counts both come from this table.

=== DOMAIN GLOSSARY ===
  'revenue' = SUM(products.price * orders.quantity), there is no 'revenue' column
  'enterprise customer' = customers.tier = 'enterprise'
  'refund / refunded' = orders.status = 'refunded'
  'shipped' = orders.status = 'shipped'
  'pending' = orders.status = 'pending'


## Step 3 — Turning each card into scoreable text

Before we can pick the relevant tables, we need **text that represents** each card. We do this by joining the table name + its column list + its description. This text is later compared against the user's question.

The `_tokens()` function turns text into a *set of words* (lowercase, split on punctuation/spaces). That's the raw material for our simple similarity score.


In [4]:
def _tokens(text):
    """Extract all words/numbers from text (lowercased).

    Example: 'Total Revenue?' -> {'total', 'revenue'}
    """
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def _card_text(table, card):
    """Text representation of one schema card: table name + columns + description."""
    return table + " " + " ".join(card["columns"]) + " " + card["doc"]


# Example: text representation for the 'products' card
product_text = _card_text("products", SCHEMA_CARDS["products"])
print("Text representation of the 'products' card:")
print(f"  {product_text}")
print(f"\nUnique tokens (words) inside it:")
print(f"  {sorted(_tokens(product_text))}")


Text representation of the 'products' card:
  products id name category price One row per product: name, category, and unit price. Revenue is computed from price here times quantity in orders.

Unique tokens (words) inside it:
  ['and', 'category', 'computed', 'from', 'here', 'id', 'in', 'is', 'name', 'one', 'orders', 'per', 'price', 'product', 'products', 'quantity', 'revenue', 'row', 'times', 'unit']


## Step 4 — Word-overlap similarity score

We use a simple score: **how many words the question and the card text have in common**. The more words overlap, the higher the score.

> **In a production system**, this is replaced by **embeddings** (e.g. from `sentence-transformers`) that measure similarity in *meaning*, not just shared words. For learning the concept, keyword overlap is enough — and it gives **the same final answer** for our example.


In [5]:
def score_card(question, table, card):
    """Simple similarity score: number of words shared between the question
    and the card's text.

    Example:
        question = 'total revenue shipped orders'
        card text for 'orders' = 'orders id customer_id product_id ... shipped ...'
        score = number of words that appear in both
    """
    q_tokens = _tokens(question)
    card_tokens = _tokens(_card_text(table, card))
    return len(q_tokens & card_tokens)


# Try it: score every card against a question about revenue
question = "What was our total revenue from shipped orders?"
print(f"Question: '{question}'\n")
print("Score per table:")
for table, card in SCHEMA_CARDS.items():
    score = score_card(question, table, card)
    print(f"  {table:<10} -> score = {score}")


Question: 'What was our total revenue from shipped orders?'

Score per table:
  customers  -> score = 0
  products   -> score = 3
  orders     -> score = 4


## Step 5 — Retrieve schema: fetch only the relevant subset

This is the **schema linking** step. We score every table, then keep the **top-k** highest scores. Why only a subset?

- A real database can have **hundreds of tables** and tens of thousands of columns. Pasting the entire schema into a prompt would be expensive and confusing for the model.
- An LLM's ability to *reason* over tabular data degrades as the number of tables grows — **long before** the context window limit is reached.

For a question about *revenue*, `k=2` is enough: we need `orders` and `products` to JOIN and SUM.


In [6]:
def retrieve_schema(question, k=2):
    """Score every schema card, return the top-k with the highest scores.

    Returns:
        List of (table_name, score, card_dict), sorted by score descending.
    """
    scored = []
    for table, card in SCHEMA_CARDS.items():
        scored.append((table, score_card(question, table, card), card))
    # Sort from highest score to lowest
    scored.sort(key=lambda item: -item[1])
    return scored[:k]


def render_schema(retrieved):
    """Format the selected schema so it's easy to read (or paste into an LLM prompt)."""
    lines = []
    for table, _score, card in retrieved:
        cols = ", ".join(card["columns"])
        lines.append(f"TABLE {table}({cols})  -- {card['doc']}")
    return "\n".join(lines)


# Retrieve the top-2 tables for the revenue question
question = "What was our total revenue from shipped orders?"
retrieved = retrieve_schema(question, k=2)

print(f"Question: '{question}'\n")
print(f"Selected tables (top-2): {[t for t, _, _ in retrieved]}\n")
print("Retrieved schema:")
print(render_schema(retrieved))


Question: 'What was our total revenue from shipped orders?'

Selected tables (top-2): ['orders', 'products']

Retrieved schema:
TABLE orders(id, customer_id, product_id, quantity, status, order_date)  -- One row per order: which customer bought which product, how many, the status (shipped, refunded, pending), and the date. Revenue and refund counts both come from this table.
TABLE products(id, name, category, price)  -- One row per product: name, category, and unit price. Revenue is computed from price here times quantity in orders.


## Step 6 — Generate SQL (rule-based version)

In a real system, this step sends **(question + selected schema + glossary + example SQL)** to an LLM and parses the SQL it returns. Here we use a **rule-based stub** — simple pattern matching — so the notebook stays lightweight and doesn't need a model. Even so, the **SQL it produces is real SQL** that actually runs.

Two things worth understanding:

1. **Match order matters.** More specific rules (revenue, refund, top-product) are checked **before** the generic "how many customers" rule. This stops a question like "how many orders were refunded?" from accidentally matching the "how many customers" branch.
2. **Guard clauses.** The *revenue* branch requires **both** `orders` **and** `products` to be present in the retrieved schema. If not, the function **refuses and returns `None`** — better to refuse than to guess a wrong number.

> We've also added a few extra query patterns below: averages, top-N, revenue by category, and date filters.


In [7]:
def generate_sql(question, retrieved):
    """Generate SQL from a question + the retrieved schema.

    Uses simple pattern matching (rule-based). In production this step is
    done by an LLM. Returns None if no pattern matches or if a required
    table is missing from the retrieved schema.
    """
    q = question.lower()
    tables = {t for t, _, _ in retrieved}

    # Rules are ordered from MOST SPECIFIC to most general.
    # If a specific pattern were checked later, a more general rule could
    # swallow it by mistake.

    # --- 1. "top product by revenue" -> GROUP BY + ORDER BY + LIMIT ---
    #        (Must be checked BEFORE the plain revenue rule, since it also
    #         contains the word 'revenue'.)
    if ("top" in q or "best selling" in q) and {"orders", "products"} <= tables:
        return ("SELECT p.name, SUM(p.price * o.quantity) AS revenue "
                "FROM orders o JOIN products p ON o.product_id = p.id "
                "WHERE o.status = 'shipped' "
                "GROUP BY p.name ORDER BY revenue DESC LIMIT 1;")

    # --- 2. "revenue by category" -> GROUP BY category ---
    if "revenue" in q and "categor" in q and {"orders", "products"} <= tables:
        return ("SELECT p.category, SUM(p.price * o.quantity) AS revenue "
                "FROM orders o JOIN products p ON o.product_id = p.id "
                "WHERE o.status = 'shipped' "
                "GROUP BY p.category ORDER BY revenue DESC;")

    # --- 3. "average order value" -> AVG(price * quantity) ---
    if ("average" in q or "avg" in q) and "order" in q and {"orders", "products"} <= tables:
        return ("SELECT AVG(p.price * o.quantity) "
                "FROM orders o JOIN products p ON o.product_id = p.id "
                "WHERE o.status = 'shipped';")

    # --- 4. "total revenue" -> JOIN + SUM(price * quantity) ---
    if "revenue" in q and {"orders", "products"} <= tables:
        # Detect a status filter if one is mentioned
        if "shipped" in q:
            return ("SELECT SUM(p.price * o.quantity) "
                    "FROM orders o JOIN products p ON o.product_id = p.id "
                    "WHERE o.status = 'shipped';")
        return ("SELECT SUM(p.price * o.quantity) "
                "FROM orders o JOIN products p ON o.product_id = p.id;")

    # --- 5. "how many refunded orders" -> COUNT with a status filter ---
    if "refund" in q and "orders" in tables:
        return "SELECT COUNT(*) FROM orders WHERE status = 'refunded';"

    # --- 6. "how many orders in march/april" -> COUNT with a date filter ---
    if ("how many" in q or "number of" in q) and "orders" in tables:
        if "march" in q:
            return ("SELECT COUNT(*) FROM orders "
                    "WHERE order_date >= '2026-03-01' AND order_date < '2026-04-01';")
        if "april" in q:
            return ("SELECT COUNT(*) FROM orders "
                    "WHERE order_date >= '2026-04-01' AND order_date < '2026-05-01';")

    # --- 7. "how many <tier> customers" -> COUNT with a tier filter ---
    if "how many" in q and "customers" in tables:
        for tier in ("enterprise", "pro", "free"):
            if tier in q:
                return f"SELECT COUNT(*) FROM customers WHERE tier = '{tier}';"
        return "SELECT COUNT(*) FROM customers;"

    # --- 8. "how many products in each category" -> COUNT with GROUP BY ---
    if "how many" in q and "categor" in q and "products" in tables:
        return "SELECT category, COUNT(*) FROM products GROUP BY category;"

    # No pattern matched, or a required table is missing.
    # Better to refuse (return None) than to guess and produce wrong SQL.
    return None


# Try it: generate SQL for the revenue question
sql = generate_sql("What was our total revenue from shipped orders?", retrieved)
print("Generated SQL:")
print(f"  {sql}")


Generated SQL:
  SELECT SUM(p.price * o.quantity) FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped';


## Step 7 — Execute & Answer: the full pipeline

This is the step that **doesn't exist** in a document pipeline: we actually **run the SQL** against a real SQLite database. The database computes the number **exactly**, so the model never has to guess.

The `sql_rag()` function is the complete pipeline:
**retrieve schema → generate SQL → execute → answer.**

If `generate_sql()` refuses (returns `None`), we say so clearly — instead of running a broken or wrong query.


In [8]:
def answer_from_rows(sql, rows):
    """Turn query results (a list of tuples) into a one-line answer.

    In production, the result rows are sent back to an LLM to phrase in
    natural language. Here we simply format the scalar value or the top row.
    """
    if not rows:
        return "No matching rows."
    # If the result has multiple columns (e.g. a top-N query), show them all
    if len(rows[0]) > 1:
        return " | ".join(str(v) for v in rows[0])
    return f"{rows[0][0]}"


def sql_rag(db, question, k=2):
    """The full SQL RAG pipeline.

    Args:
        db: an sqlite3 connection.
        question: the user's question in natural language.
        k: how many tables to retrieve (default 2).

    Returns:
        dict with: tables (retrieved), sql (generated), answer.
    """
    # 1. Retrieve — pick the relevant tables
    retrieved = retrieve_schema(question, k=k)

    # 2. Generate — write SQL based on the selected schema
    sql = generate_sql(question, retrieved)

    # 3. If no SQL could be generated, refuse politely
    if sql is None:
        return {
            "tables": [t for t, _, _ in retrieved],
            "sql": None,
            "answer": "I can't translate this question into SQL.",
        }

    # 4. Execute — run the SQL against the real database
    rows = db.execute(sql).fetchall()

    # 5. Answer — format the result
    return {
        "tables": [t for t, _, _ in retrieved],
        "sql": sql,
        "answer": answer_from_rows(sql, rows),
    }


# End-to-end test
result = sql_rag(db, "What was our total revenue from shipped orders?")
print("Retrieved tables:", result["tables"])
print("SQL:", result["sql"])
print("Answer:", result["answer"])
print()
print("Manual check: shipped orders =")
print("  order 100: 129 * 2 = 258 (Nimbus Speaker)")
print("  order 101:  49 * 1 =  49 (Aurora Lamp)")
print("  order 103: 129 * 1 = 129 (Nimbus Speaker)")
print("  order 105:  19 * 4 =  76 (Solaris Bulb)")
print("  order 106: 129 * 1 = 129 (Nimbus Speaker)")
print("  TOTAL     = 641.0")


Retrieved tables: ['orders', 'products']
SQL: SELECT SUM(p.price * o.quantity) FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped';
Answer: 641.0

Manual check: shipped orders =
  order 100: 129 * 2 = 258 (Nimbus Speaker)
  order 101:  49 * 1 =  49 (Aurora Lamp)
  order 103: 129 * 1 = 129 (Nimbus Speaker)
  order 105:  19 * 4 =  76 (Solaris Bulb)
  order 106: 129 * 1 = 129 (Nimbus Speaker)
  TOTAL     = 641.0


## Step 8 — Router: SQL or documents?

Now we have **two kinds of pipeline**: a document pipeline (from a text-RAG system elsewhere) and this SQL pipeline. We need a **router** that decides which one a question should go through.

Simple rule: if the question contains an **aggregation/number signal** (`how many`, `count`, `total`, `sum`, `average`, `top`, `revenue`), send it to **SQL**. Everything else (e.g. *"what's our refund policy?"*) goes to the **document** pipeline.

> **Watch out for:** the dangerous direction is a *silent misroute* — a numeric question that ends up going to documents. The model will find a paragraph that **sounds similar**, then **make up a number**. So when in doubt, lean toward SQL: a wrong SQL query usually fails clearly, while a made-up number does not.


In [9]:
SQL_SIGNALS = (
    "how many", "count", "total", "sum", "average", "avg",
    "most", "top", "revenue", "per ", "number of", "category",
)


def route(question):
    """Return 'sql' or 'documents' based on keyword signals in the question."""
    q = question.lower()
    if any(signal in q for signal in SQL_SIGNALS):
        return "sql"
    return "documents"


# Test with a few different styles of question
examples = [
    "How many enterprise customers do we have?",
    "What is your refund policy?",
    "What is the total revenue by category?",
    "What are your shipping terms?",
]
for q in examples:
    print(f"  route = {route(q):<9} <- {q}")


  route = sql       <- How many enterprise customers do we have?
  route = documents <- What is your refund policy?
  route = sql       <- What is the total revenue by category?
  route = documents <- What are your shipping terms?


## Step 9 — Demo: several questions, two pipelines

Now let's run a batch of questions through the router. Aggregation-style questions will go through the SQL pipeline — retrieve schema → generate SQL → execute → answer. Descriptive questions will be routed to the document pipeline (which is just a stub here, since we're focused on the SQL side).

At the end we double-check that every SQL answer actually matches the data we seeded.


In [10]:
DEMO = [
    "How many enterprise customers do we have?",
    "What was our total revenue from shipped orders?",
    "How many orders were refunded?",
    "What is the average order value for shipped orders?",
    "What is the top product by revenue?",
    "How many orders were placed in March?",
    "What is your refund policy?",   # this will be routed to the document pipeline
]

for q in DEMO:
    r = route(q)
    print(f"Q: {q}")
    print(f"   route = {r}")
    if r != "sql":
        print("   -> document pipeline: retrieve passages, ground, generate.")
        print()
        continue
    out = sql_rag(db, q)
    print(f"   retrieved schema : {out['tables']}")
    print(f"   SQL              : {out['sql']}")
    print(f"   answer           : {out['answer']}")
    print()

# --- Automatic check: do the SQL answers match the seeded data? ---
expected = {
    "How many enterprise customers do we have?": "2",    # Bashir, Eka
    "What was our total revenue from shipped orders?": "641.0",
    "How many orders were refunded?": "1",               # order 102
    "How many orders were placed in March?": "5",        # orders 100-104
}
print("=== Verification ===")
for q, expected_answer in expected.items():
    actual = sql_rag(db, q)["answer"]
    status = "OK" if actual == expected_answer else f"WRONG (got {actual})"
    print(f"  [{status}] {q} -> expected={expected_answer}")


Q: How many enterprise customers do we have?
   route = sql
   retrieved schema : ['customers', 'orders']
   SQL              : SELECT COUNT(*) FROM customers WHERE tier = 'enterprise';
   answer           : 2

Q: What was our total revenue from shipped orders?
   route = sql
   retrieved schema : ['orders', 'products']
   SQL              : SELECT SUM(p.price * o.quantity) FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped';
   answer           : 641.0

Q: How many orders were refunded?
   route = sql
   retrieved schema : ['orders', 'products']
   SQL              : SELECT COUNT(*) FROM orders WHERE status = 'refunded';
   answer           : 1

Q: What is the average order value for shipped orders?
   route = sql
   retrieved schema : ['orders', 'products']
   SQL              : SELECT AVG(p.price * o.quantity) FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped';
   answer           : 128.2

Q: What is the top product by revenue

## Step 10 — Breaking it on purpose: a schema-linking failure

The idea sinks in best when something actually **fails**. `retrieve_schema()` fetches the top-`k` tables. Try lowering `k` to **1** and repeat the revenue question. Now only the `orders` table is retrieved; the `products` table (which stores `price`) never makes it in. The guard clause inside `generate_sql()` will **refuse** — not guess.

This is a **correct failure**: the query needs two tables but only got one, so it refuses. The production lesson: **on structured data, an incomplete schema retrieval is a wrong answer (or no answer), not a "smaller" one.**

> **Most wrong answers on structured data come from schema-linking failures**, not from generation failures.


In [11]:
result_k1 = sql_rag(db, "What was our total revenue from shipped orders?", k=1)
print("k=1 retrieved:", result_k1["tables"])
print("k=1 SQL     :", result_k1["sql"])
print("k=1 answer  :", result_k1["answer"])
print()
print("The system refuses instead of guessing. That's the correct failure.")


k=1 retrieved: ['orders']
k=1 SQL     : None
k=1 answer  : I can't translate this question into SQL.

The system refuses instead of guessing. That's the correct failure.


## Step 11 — Extension: more advanced queries

Let's try a few more advanced queries to see the same pipeline flex its muscles:

- **AVG** (average order value)
- **GROUP BY + ORDER BY + LIMIT** (best-selling product)
- **GROUP BY category** (revenue split by product category — new in this version)
- **GROUP BY count** (how many products per category — new in this version)
- **Date filters** (orders in a given month)

Since `generate_sql()` already has patterns for all of these, we just call `sql_rag()`.


In [12]:
advanced_questions = [
    "What is the average order value for shipped orders?",
    "What is the top product by revenue?",
    "What is the total revenue by category?",
    "How many products are there in each category?",
    "How many orders were placed in April?",
    "How many orders were placed in March?",
]

for q in advanced_questions:
    result = sql_rag(db, q)
    print(f"Q: {q}")
    print(f"   tables : {result['tables']}")
    print(f"   SQL    : {result['sql']}")
    print(f"   answer : {result['answer']}")
    print()


Q: What is the average order value for shipped orders?
   tables : ['orders', 'products']
   SQL    : SELECT AVG(p.price * o.quantity) FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped';
   answer : 128.2

Q: What is the top product by revenue?
   tables : ['products', 'orders']
   SQL    : SELECT p.name, SUM(p.price * o.quantity) AS revenue FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped' GROUP BY p.name ORDER BY revenue DESC LIMIT 1;
   answer : Nimbus Speaker | 516.0

Q: What is the total revenue by category?
   tables : ['products', 'orders']
   SQL    : SELECT p.category, SUM(p.price * o.quantity) AS revenue FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'shipped' GROUP BY p.category ORDER BY revenue DESC;
   answer : audio | 516.0

Q: How many products are there in each category?
   tables : ['products', 'orders']
   SQL    : SELECT category, COUNT(*) FROM products GROUP BY category;
   answer : au

## Step 12 — Exploring the raw data

Sometimes it helps to look directly at what's in the database (for example, to sanity-check a SQL answer). The `show_table()` helper below prints the full contents of any table in a neat, column-aligned format.


In [13]:
def show_table(db, table_name, limit=20):
    """Print the contents of a table as neatly aligned columns."""
    # Get the column names from PRAGMA
    cols_info = db.execute(f"PRAGMA table_info({table_name})").fetchall()
    columns = [c[1] for c in cols_info]

    rows = db.execute(f"SELECT * FROM {table_name} LIMIT {limit}").fetchall()

    # Print the header
    header = " | ".join(f"{c:<12}" for c in columns)
    print(header)
    print("-" * len(header))
    for row in rows:
        print(" | ".join(f"{str(v):<12}" for v in row))


for table_name in ["customers", "products", "orders"]:
    print(f"\n=== {table_name.upper()} ===")
    show_table(db, table_name)



=== CUSTOMERS ===
id           | name         | country      | tier        
---------------------------------------------------------
1            | Ada          | DE           | pro         
2            | Bashir       | TR           | enterprise  
3            | Chen         | SG           | free        
4            | Dilan        | TR           | pro         
5            | Eka          | ID           | enterprise  
6            | Farid        | ID           | free        

=== PRODUCTS ===
id           | name         | category     | price       
---------------------------------------------------------
10           | Aurora Lamp  | lighting     | 49.0        
11           | Nimbus Speaker | audio        | 129.0       
12           | Coil Cable   | audio        | 9.0         
13           | Solaris Bulb | lighting     | 19.0        

=== ORDERS ===
id           | customer_id  | product_id   | quantity     | status       | order_date  
---------------------------------------------

## Step 13 — Try your own question

Edit the `your_question` variable below and run the cell. Try patterns the pipeline already supports:

- *"How many free customers?"*
- *"How many orders were placed in April?"*
- *"What is the top product by revenue?"*
- *"What is the average order value for shipped orders?"*
- *"What is the total revenue by category?"*
- *"How many products are there in each category?"*

Or try a question that's **not supported yet** and watch the system refuse politely — that's exactly the right behavior for a production system.


In [14]:
your_question = "How many pro customers do we have?"

result = sql_rag(db, your_question)
print(f"Question: {your_question}\n")
print(f"Retrieved tables : {result['tables']}")
print(f"Generated SQL    : {result['sql']}")
print(f"Answer           : {result['answer']}")


Question: How many pro customers do we have?

Retrieved tables : ['customers', 'orders']
Generated SQL    : SELECT COUNT(*) FROM customers WHERE tier = 'pro';
Answer           : 2


## Summary — what you learned

1. **Dense passage retrieval can't answer questions that require computation.** A number like "total revenue" doesn't exist as text until a query computes it. Retrieving the closest paragraph just produces a **made-up number**. The fix is a query, not a better embedder.

2. **Text-to-SQL RAG** swaps two steps in the document pipeline: from **embed → retrieve → ground → generate** to **retrieve schema → generate SQL → execute → answer**. The database does the arithmetic exactly, so the model never has to guess.

3. **What gets retrieved isn't just table names**: **schema cards** (with column descriptions and example rows) plus a **domain glossary** (mapping business terms like "revenue" to columns + formulas). The glossary and example rows prevent most "invented column" and "wrong format" errors.

4. **Retrieve a subset of the schema (schema linking)**, not the whole catalog. Why: (a) real schemas can be bigger than the context window; (b) an LLM's ability to *reason* over tables degrades as the number of tables grows, well before any token limit is hit.

5. **Routing** sends aggregation/numeric questions to SQL, and everything else to the document pipeline. Lean slightly toward SQL for borderline cases — misrouting to documents fails **silently** (a made-up number), while misrouting to SQL usually fails **loudly** (a query error or a refusal).

6. **Guard clauses and "refusing politely".** A system that declines to answer when the schema is incomplete is better than one that produces a made-up number. On structured data, a wrong answer is more dangerous than no answer.

7. **Everything runs offline.** The execution step is real (actual SQL against a real SQLite database), while schema retrieval (keyword overlap) and SQL generation (rule-based) are simplified so the notebook needs no extra dependencies. The control flow is identical to a production system.

### Next steps for a production system

- Replace keyword overlap with **sentence-transformer embeddings** for scoring schema cards.
- Replace the rule-based `generate_sql()` with an **LLM**, prompted with: question + schema cards + glossary + a few example SQL queries.
- Add **query validation** (e.g. `EXPLAIN QUERY PLAN`, syntax checks, restricting which tables can be touched).
- Add **result verification** — a second LLM pass that checks the answer before it's shown to the user.
- Add **logging** for every generated SQL statement — essential for auditing and debugging.

---

**© Patria & Co. 2026** · AI & Data Science Consultancy
🌐 [www.patriaco.co.uk](https://www.patriaco.co.uk)
